In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("Groq Key Loaded:", GROQ_API_KEY is not None)


Groq Key Loaded: True


In [3]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)


c:\Users\mihir\Documents\Python Scripts\MajorProj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from llama_index.llms.groq import Groq

Settings.llm = Groq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.0
)


In [5]:
import chromadb
persist_dir = "./chroma_db"

chroma_client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=chromadb.Settings(anonymized_telemetry=False)
)

collection = chroma_client.get_or_create_collection("rag_collection")



In [6]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [7]:
import fitz
from llama_index.core import Document
from llama_index.core.schema import TextNode
from llama_index.core.node_parser import SentenceSplitter

def translate_chunk(text_en):
    prompt = f"Translate this technical text to Chinese while preserving terminology:\n\n{text_en}"
    response = Settings.llm.complete(prompt)
    return str(response)

def process_aligned_ingestion(file_path):
    #Extraction
    doc = fitz.open(file_path)
    full_en_text = "\n".join([page.get_text() for page in doc])

    #Aligned Chunking
    en_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
    en_chunks = en_splitter.split_text(full_en_text)
    
    nodes = []
    
    for en_chunk in en_chunks:
        # Translate each chunk
        node = TextNode(
            text=en_chunk,
            metadata={
            "file_name": file_path.split("/")[-1]
            }
        )
        nodes.append(node)
    return nodes


In [11]:
from llama_index.core import VectorStoreIndex
from pathlib import Path

DATA_DIR = "data"
all_nodes = []

for pdf in Path(DATA_DIR).glob("*.pdf"):
    nodes = process_aligned_ingestion(str(pdf))
    all_nodes.extend(nodes)

index = VectorStoreIndex(
    all_nodes,
    storage_context=storage_context
)

# ------------------------
# Query stage
# ------------------------

query = input("Enter your question: ")
retriever = index.as_retriever(similarity_top_k=3)
nodes = retriever.retrieve(query)
contexts = []

for node in nodes:

    en_text = node.text
    zh_context = translate_chunk(en_text)

    contexts.append(
        f"English Context:\n{en_text}\n\nChinese Context:\n{zh_context}"
    )

context_str = "\n\n".join(contexts)

prompt = f"""
Use the Chinese context for reasoning but answer in English.
{context_str}
Question: {query}
Answer:
"""
response = Settings.llm.complete(prompt)
print(response)

According to the Chinese context, Zeus and Poseidon did not marry Thetis because they were afraid that their son would be greater than them and eventually oust them from their seats of power. This fear was based on a prophecy that the son born to Thetis would be more powerful than his father.


In [ ]:
#Prototype AGENT, dont use, functinons might be deprecated

from llama_index.core.agent import ReActAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata

query_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="document_expert",
        description="Extracts info from the document using English/Chinese context."
    )
)
# ReAct Agent with system prompt
agent = ReActAgent(
    tools=[query_tool],
    llm=Settings.llm,
    verbose=True,
    # System prompt is passed directly to the constructor
    system_prompt=(
        "You are an expert analyst. Always prioritize the 'zh_context' "
        "in metadata for your internal reasoning, but answer in English."
    )
)

response =  await agent.run("Based on the document, is the stem soft? Verify using the Chinese context.")
print(response)

Yes, the stem of the common sunflower is soft in the sense that it is rough-hairy, but it is not a soft stem in the sense that it is not a flexible or pliable stem, rather it is a stiff stem that can reach heights of 3 meters.
